# Compute Embeddings Database

This notebook created by [biodiversica](https://biodiversica.xyz) computes **audio embeddings** from a **foundation model backbone** (ONNX format) and saves them to a **SQLite database** on your Google Drive.

---

### What this notebook does (in order):
1. **Connects to your Google Drive** to read audio and save results
2. **Installs the necessary software** automatically
3. **Loads audio** from Google Drive or an Arbimon project
4. **Loads a backbone model** from HuggingFace (Perch v2, BirdNET, or custom ONNX)
5. **Computes embeddings** for each audio segment and saves them to a SQLite database

### Supported preset models:
| Preset | HuggingFace repo | Sample rate | Window | Embedding size |
|---|---|---|---|---|
| Perch v2.0 | `biodiversica/Perch-onnx-backbone` | 32 000 Hz | 5.0 s | 1536 |
| BirdNET v2.4 | `biodiversica/BirdNET-onnx-backbone` | 48 000 Hz | 3.0 s | 1024 |

### Output database:
Embeddings are saved to a SQLite file `{model_name}.embeddings.db` in your output folder.
Tables:
- `metadata` — model parameters (`model_id`, `sample_rate`, `window_duration_s`, `overlap`, `embedding_size`, …)
- `embeddings` — one row per segment: `(site_name, recording_id, chunk_index, datetime, data)`
- `sites` — optional per-site coordinates and stream IDs

Chunk start time is derived from `chunk_index` rather than stored directly:
`start_time = chunk_index × floor(window_duration_s × sample_rate × (1 − overlap)) / sample_rate`

### How to run:
Run each cell **one at a time**, top to bottom. Click ▶ or press `Shift + Enter`.

> **New to notebooks?** A cell with `[ ]` on the left has not been run. After running it shows `[1]`, `[2]`, etc. If you see an error (red text), read the message — it usually tells you exactly what to fix.

---
## Step 1 — Connect Google Drive & Install Software

Run the two cells below. The first will ask you to **allow access to your Google Drive** — click the link and follow the steps.

In [ ]:
#@title 📂 Connect Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected successfully.')

In [ ]:
#@title 📦 Install packages { display-mode: "form" }

!wget -q https://github.com/rfcx/rfcx-sdk-python/releases/download/0.3.1/rfcx-0.3.1-py3-none-any.whl -O /tmp/rfcx-0.3.1-py3-none-any.whl

!pip install /tmp/rfcx-0.3.1-py3-none-any.whl -q
!pip install onnxruntime librosa soundfile huggingface_hub -q

print('\nAll packages installed successfully.')

---
## Step 2 — Configuration

Fill in the forms below. **You do not need to edit any code** — just type or select your values in each form and run the cell.

Run all three forms from top to bottom:
1. **General Settings** — output folder, overlap
2. **Audio Source** — where your recordings are stored
3. **Model** — which backbone to use

> **Tip:** The forms hide the code automatically. To see the underlying code, click the `{ }` icon in the top-right corner of any form cell.

In [ ]:
#@title ⚙️ General Settings { display-mode: "form" }

import os

#@markdown **Embeddings output folder** — where the SQLite database will be saved on your Google Drive.
DRIVE_EMBEDDINGS_DIR = '/content/drive/MyDrive/embeddings'  #@param {type:"string"}

#@markdown ---
#@markdown **Segment overlap (0.0–0.9)** — fraction of overlap between consecutive audio windows.
#@markdown 0.0 = no overlap (default). 0.5 = 50% overlap (denser coverage, twice as many segments per file).
SEGMENT_OVERLAP = 0.0  #@param {type:"slider", min:0.0, max:0.9, step:0.1}

#@markdown ---
#@markdown **Database mode** — create a new database or update an existing one.
#@markdown Use *update existing* to add recordings to a database you already built.
DB_MODE = 'create new'  #@param ["create new", "update existing"]

#@markdown **Existing database path** — full path to an existing `.db` file on your Google Drive.
#@markdown Only used when **Database mode = update existing**.
EXISTING_DB_PATH = '/content/drive/MyDrive/embeddings/my_model.embeddings.db'  #@param {type:"string"}

os.makedirs(DRIVE_EMBEDDINGS_DIR, exist_ok=True)
print(f'Embeddings folder : {DRIVE_EMBEDDINGS_DIR}')
print(f'Segment overlap   : {SEGMENT_OVERLAP}')

In [ ]:
#@title 🗂️ Audio Source { display-mode: "form" }

import os

#@markdown **Source** — where your audio files are stored.
AUDIO_SOURCE = 'google_drive'  #@param ["google_drive", "arbimon"]

#@markdown ---
#@markdown ### Google Drive settings *(used when source = google_drive)*
#@markdown Full path to the folder containing your audio recordings on Google Drive.
DRIVE_INPUT_DIR = '/content/drive/MyDrive/audio'  #@param {type:"string"}

#@markdown **Filename prefix** — leave blank for standard AudioMoth files (`YYYYMMDD_HHMMSS.wav`).
#@markdown Enter `SM4` for files like `SM4_20250615_203000.wav`.
FILENAME_PREFIX = ''  #@param {type:"string"}

#@markdown **Date/time filter** — only process files within this date/time range.
#@markdown Leave disabled to process all files.
FILTER_ENABLED = False  #@param {type:"boolean"}
FILTER_START_DATE = "2025-01-01"  #@param {type:"date"}
FILTER_START_TIME = "00:00"  #@param {type:"string"}
FILTER_END_DATE = "2025-12-31"  #@param {type:"date"}
FILTER_END_TIME = "23:59"  #@param {type:"string"}

#@markdown ---
#@markdown **Site coordinates** (optional) — latitude and longitude per recording site.
#@markdown Format: `SITE_NAME=lat,lon` separated by semicolons.
#@markdown Example: `POINT_A=-15.1,-47.2;POINT_B=-16.3,-48.1`
SITE_COORDINATES = ''  #@param {type:"string"}

FILENAME_PREFIX = FILENAME_PREFIX.strip()
_latlon_map = {}
for _e in SITE_COORDINATES.split(';'):
    _e = _e.strip()
    if '=' in _e:
        _n, _c = _e.split('=', 1)
        _p = _c.split(',')
        try:
            _latlon_map[_n.strip()] = (float(_p[0]), float(_p[1]))
        except Exception:
            pass

print(f'Audio source : {AUDIO_SOURCE}')
if AUDIO_SOURCE == 'google_drive':
    if not os.path.isdir(DRIVE_INPUT_DIR):
        print(f'WARNING: Folder not found: {DRIVE_INPUT_DIR}')
        print('Check the path above — make sure Google Drive is connected.')
    else:
        print(f'Audio folder : {DRIVE_INPUT_DIR}')
    if FILTER_ENABLED:
        print(f'Date filter  : {FILTER_START_DATE} {FILTER_START_TIME} → {FILTER_END_DATE} {FILTER_END_TIME}')
    else:
        print('Date filter  : disabled (all files will be processed)')
elif AUDIO_SOURCE == 'arbimon':
    print('Arbimon source selected — configure recording sites in Step 3.')

In [ ]:
#@title 🤖 Model { display-mode: "form" }

import os

#@markdown **Model preset** — select a foundation model backbone.
#@markdown - **Perch v2**: Google Perch backbone (32 kHz · 5 s window · 1536-dim embeddings)
#@markdown - **BirdNET backbone**: BirdNET v2.4 backbone (48 kHz · 3 s window · 1024-dim embeddings)
#@markdown - **Custom ONNX**: provide your own ONNX backbone model
MODEL_PRESET = 'Perch v2'  #@param ["Perch v2", "BirdNET backbone", "Custom ONNX"]

#@markdown ---
#@markdown ### Custom ONNX settings *(used when preset = Custom ONNX)*

#@markdown **Custom model source**
CUSTOM_MODEL_SOURCE = 'google_drive'  #@param ["google_drive", "huggingface"]

#@markdown **Google Drive path** — full path to your `.onnx` backbone file on Drive.
CUSTOM_DRIVE_MODEL_PATH = '/content/drive/MyDrive/models/my_backbone.onnx'  #@param {type:"string"}

#@markdown **HuggingFace repo ID** — e.g. `my_org/my-backbone-onnx`
CUSTOM_HF_REPO = ''  #@param {type:"string"}

#@markdown **HuggingFace filename** — e.g. `backbone.onnx`
CUSTOM_HF_FILE = 'backbone.onnx'  #@param {type:"string"}

#@markdown **Sample rate (Hz)**
CUSTOM_SAMPLE_RATE = 48000  #@param {type:"integer"}

#@markdown **Window duration (seconds)** — length of each audio window fed to the model.
CUSTOM_WINDOW_S = 3.0  #@param {type:"number"}

#@markdown **ONNX input node name**
CUSTOM_INPUT_NAME = 'input'  #@param {type:"string"}

#@markdown **ONNX output node name**
CUSTOM_OUTPUT_NAME = 'embedding'  #@param {type:"string"}

#@markdown **Embedding size** — number of dimensions in the output vector.
CUSTOM_EMBEDDING_SIZE = 1024  #@param {type:"integer"}

_PRESETS = {
    'Perch v2': {
        'hf_repo':        'biodiversica/Perch-onnx-backbone',
        'hf_file':        'perch_v2_backbone.onnx',
        'sample_rate':    32000,
        'window_s':       5.0,
        'input_name':     'inputs',
        'output_name':    'embedding',
        'embedding_size': 1536,
        'model_name':     'perch_v2_backbone',
    },
    'BirdNET backbone': {
        'hf_repo':        'biodiversica/BirdNET-onnx-backbone',
        'hf_file':        'model_backbone.onnx',
        'sample_rate':    48000,
        'window_s':       3.0,
        'input_name':     'INPUT',
        'output_name':    'embedding',
        'embedding_size': 1024,
        'model_name':     'model_backbone',
    },
}

if MODEL_PRESET in _PRESETS:
    _p = _PRESETS[MODEL_PRESET]
    HF_REPO        = _p['hf_repo']
    HF_FILE        = _p['hf_file']
    SAMPLE_RATE    = _p['sample_rate']
    WINDOW_S       = _p['window_s']
    INPUT_NAME     = _p['input_name']
    OUTPUT_NAME    = _p['output_name']
    EMBEDDING_SIZE = _p['embedding_size']
    MODEL_NAME     = _p['model_name']
else:
    SAMPLE_RATE    = CUSTOM_SAMPLE_RATE
    WINDOW_S       = CUSTOM_WINDOW_S
    INPUT_NAME     = CUSTOM_INPUT_NAME
    OUTPUT_NAME    = CUSTOM_OUTPUT_NAME
    EMBEDDING_SIZE = CUSTOM_EMBEDDING_SIZE
    if CUSTOM_MODEL_SOURCE == 'huggingface':
        HF_REPO    = CUSTOM_HF_REPO.strip()
        HF_FILE    = CUSTOM_HF_FILE.strip()
        MODEL_NAME = os.path.splitext(HF_FILE)[0]
    else:
        HF_REPO    = None
        HF_FILE    = None
        MODEL_NAME = os.path.splitext(os.path.basename(CUSTOM_DRIVE_MODEL_PATH))[0]

WINDOW_SAMPLES = int(WINDOW_S * SAMPLE_RATE)

print(f'Model preset    : {MODEL_PRESET}')
print(f'Model name      : {MODEL_NAME}')
print(f'Sample rate     : {SAMPLE_RATE} Hz')
print(f'Window duration : {WINDOW_S} s  ({WINDOW_SAMPLES} samples)')
print(f'Embedding size  : {EMBEDDING_SIZE}')
print(f'Overlap         : {SEGMENT_OVERLAP}')
if HF_REPO:
    print(f'HF repo         : {HF_REPO}')
    print(f'HF file         : {HF_FILE}')

---
## Step 3 — Log in to Arbimon *(skip if using Google Drive)*

Run the cells below **only if** you set `AUDIO_SOURCE = "arbimon"` in Step 2.
If you are using Google Drive, jump directly to **Step 4**.

**What to expect on first login:** a URL and a short code will appear. Open that URL in your browser and enter the code to authorize access. Your credentials are saved to Google Drive for future runs.

In [ ]:
#@title 🔑 Log in to Arbimon { display-mode: "form" }
CREDENTIALS_PATH = '/content/drive/MyDrive/rfcx/.rfcx_credentials'  #@param {type:"string"}

import os
import rfcx

os.makedirs(os.path.dirname(CREDENTIALS_PATH), exist_ok=True)

client = rfcx.Client()

if os.path.exists(CREDENTIALS_PATH):
    print('Credentials file found — logging in automatically...')
    client.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
else:
    print('No saved credentials found.')
    print('A URL will appear below. Open it in your browser and log in with your Arbimon account.')
    print('-' * 70)
    client.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    print('-' * 70)
    print(f'Credentials saved to: {CREDENTIALS_PATH}')
    print('Next time, login will happen automatically.')

print('\nLogged in to Arbimon successfully.')

In [ ]:
#@title 🔍 List my Arbimon projects and sites { display-mode: "form" }
#@markdown Optional — run this cell to find the stream IDs for your recording sites.

print('Fetching your Arbimon projects and recording sites...')
print('=' * 70)

projects = client.projects()
if not projects:
    print('No projects found. Make sure you are logged in to the correct account.')
else:
    for project in projects:
        print(f"\nPROJECT: {project['name']}  (id: {project['id']})")
        print('  Recording sites (streams):')
        try:
            streams = client.streams(projects=project['id'])
            if streams:
                for stream in streams:
                    print(f"    stream_id: '{stream['id']}',  name: '{stream['name']}'")
            else:
                print('    (no sites found in this project)')
        except Exception as e:
            print(f'    (could not load sites: {e})')

In [ ]:
#@title 📍 Recording Site 1 { display-mode: "form" }

#@markdown **Stream ID** — the short code for this recording site (from the optional cell above).
stream_id_1 = ''  #@param {type:"string"}
#@markdown **Start date and time** (your local time)
min_date_1 = "2025-12-01"  #@param {type:"date"}
min_time_1 = "17:00"  #@param {type:"string"}
#@markdown **End date and time** (your local time)
max_date_1 = "2025-12-01"  #@param {type:"date"}
max_time_1 = "18:00"  #@param {type:"string"}
if stream_id_1.strip():
    print(f"Site 1: '{stream_id_1}' — {min_date_1} {min_time_1} → {max_date_1} {max_time_1}")
else:
    print('Site 1: no stream ID entered.')

In [ ]:
#@title ✅ Confirm recording site list { display-mode: "form" }
#@markdown Run this cell after filling in the site forms above.

import datetime

_empty = ("", "2025-01-01", "00:00", "2025-01-01", "00:00")
try: stream_id_2
except NameError: stream_id_2, min_date_2, min_time_2, max_date_2, max_time_2 = _empty
try: stream_id_3
except NameError: stream_id_3, min_date_3, min_time_3, max_date_3, max_time_3 = _empty
try: stream_id_4
except NameError: stream_id_4, min_date_4, min_time_4, max_date_4, max_time_4 = _empty

def _build_job(stream_id, min_date_str, min_time_str, max_date_str, max_time_str):
    if not stream_id.strip():
        return None
    def _parse(date_str, time_str):
        d = datetime.date.fromisoformat(date_str.strip())
        parts = time_str.strip().split(':')
        h, m = int(parts[0]), int(parts[1])
        s = int(parts[2]) if len(parts) > 2 else 0
        return datetime.datetime(d.year, d.month, d.day, h, m, s)
    return {
        'stream_id':        stream_id.strip(),
        'stream_name':      stream_id.strip(),
        'min_date':         _parse(min_date_str, min_time_str),
        'max_date':         _parse(max_date_str, max_time_str),
        'lat':              '',
        'lon':              '',
        'timezone_str':     None,
        'utc_offset_hours': 0,
    }

def _tz_to_utc_offset(timezone_value, reference_dt=None):
    if timezone_value is None or timezone_value == "":
        return None
    try:
        return float(timezone_value)
    except (ValueError, TypeError):
        pass
    try:
        from zoneinfo import ZoneInfo
        import datetime as _dt
        tz = ZoneInfo(str(timezone_value))
        ref = (reference_dt or _dt.datetime.utcnow()).replace(tzinfo=_dt.timezone.utc)
        offset = ref.astimezone(tz).utcoffset()
        return offset.total_seconds() / 3600
    except Exception:
        return None

JOBS = []
for args in [
    (stream_id_1, min_date_1, min_time_1, max_date_1, max_time_1),
    (stream_id_2, min_date_2, min_time_2, max_date_2, max_time_2),
    (stream_id_3, min_date_3, min_time_3, max_date_3, max_time_3),
    (stream_id_4, min_date_4, min_time_4, max_date_4, max_time_4),
]:
    job = _build_job(*args)
    if job:
        JOBS.append(job)

_all_streams = client.streams() or []
_stream_map = {s['id']: s for s in _all_streams if isinstance(s, dict) and 'id' in s}
for job in JOBS:
    s = _stream_map.get(job['stream_id'], {})
    job['stream_name'] = s.get('name', job['stream_id']).replace(' ', '_')
    job['lat'] = s.get('latitude') or s.get('lat', '') or ''
    job['lon'] = s.get('longitude') or s.get('lon', '') or ''
    tz_val = s.get('timezone', None)
    job['timezone_str'] = str(tz_val) if tz_val else None
    mid_dt = job['min_date'] + (job['max_date'] - job['min_date']) / 2
    utc_off = _tz_to_utc_offset(tz_val, reference_dt=mid_dt)
    if utc_off is not None:
        job['utc_offset_hours'] = utc_off
    else:
        print(f"  WARNING: no timezone found for {job['stream_name']} — defaulting to UTC+0")

if not JOBS:
    print('WARNING: No recording sites configured. Fill in at least one site form above.')
else:
    print(f'{len(JOBS)} site(s) ready:')
    for j in JOBS:
        utc_off = j['utc_offset_hours']
        print(f"  {j['stream_name']} ({j['stream_id']})")
        print(f"    {j['min_date'].strftime('%Y-%m-%d %H:%M')} → {j['max_date'].strftime('%Y-%m-%d %H:%M')}  (local UTC{utc_off:+.0f})")
    print('\nConfiguration complete. Continue to Step 4.')

### 📍 Additional Recording Sites (optional)
> Need to analyze more than one site? Expand this section to configure up to 3 more recording sites.
> Leave the **Stream ID** empty in any site you don't need. Run **Confirm recording site list** again after adding optional sites.

In [ ]:
#@title 📍 Recording Site 2 (optional) { display-mode: "form" }
#@markdown Leave **Stream ID** empty to skip this site.
stream_id_2 = ''  #@param {type:"string"}
min_date_2 = "2025-01-01"  #@param {type:"date"}
min_time_2 = "18:00"  #@param {type:"string"}
max_date_2 = "2025-01-02"  #@param {type:"date"}
max_time_2 = "06:00"  #@param {type:"string"}
if stream_id_2.strip():
    print(f"Site 2: '{stream_id_2}' — {min_date_2} {min_time_2} → {max_date_2} {max_time_2}")
else:
    print('Site 2: empty — will be skipped.')

In [ ]:
#@title 📍 Recording Site 3 (optional) { display-mode: "form" }
#@markdown Leave **Stream ID** empty to skip this site.
stream_id_3 = ''  #@param {type:"string"}
min_date_3 = "2025-01-01"  #@param {type:"date"}
min_time_3 = "18:00"  #@param {type:"string"}
max_date_3 = "2025-01-02"  #@param {type:"date"}
max_time_3 = "06:00"  #@param {type:"string"}
if stream_id_3.strip():
    print(f"Site 3: '{stream_id_3}' — {min_date_3} {min_time_3} → {max_date_3} {max_time_3}")
else:
    print('Site 3: empty — will be skipped.')

In [ ]:
#@title 📍 Recording Site 4 (optional) { display-mode: "form" }
#@markdown Leave **Stream ID** empty to skip this site.
stream_id_4 = ''  #@param {type:"string"}
min_date_4 = "2025-01-01"  #@param {type:"date"}
min_time_4 = "18:00"  #@param {type:"string"}
max_date_4 = "2025-01-02"  #@param {type:"date"}
max_time_4 = "06:00"  #@param {type:"string"}
if stream_id_4.strip():
    print(f"Site 4: '{stream_id_4}' — {min_date_4} {min_time_4} → {max_date_4} {max_time_4}")
else:
    print('Site 4: empty — will be skipped.')

---
## Step 4 — Scan / Download Audio

This cell discovers all audio files to be processed.

- **Google Drive**: scans the folder you configured in Step 2, respecting any date filter.
- **Arbimon**: downloads recordings from the sites configured in Step 3.

Files are organized by recording site. The result is summarized below — continue to Step 5 after reviewing.

In [ ]:
#@title 🔍 Scan / ⬇️ Download audio { display-mode: "form" }

import os
import re
import glob as _glob
from collections import defaultdict
from datetime import datetime

def _parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def _get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

all_audio_info = []  # list of {"path": str, "site": str, "dt": datetime|None}

if AUDIO_SOURCE == 'google_drive':
    if not os.path.isdir(DRIVE_INPUT_DIR):
        raise FileNotFoundError(f'Audio folder not found: {DRIVE_INPUT_DIR}')

    _all = sorted(
        _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.wav'),  recursive=True) +
        _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.flac'), recursive=True) +
        _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.mp3'),  recursive=True)
    )

    if FILTER_ENABLED and _all:
        _fs = datetime.strptime(f"{FILTER_START_DATE} {FILTER_START_TIME or '00:00'}", '%Y-%m-%d %H:%M')
        _fe = datetime.strptime(f"{FILTER_END_DATE} {FILTER_END_TIME or '23:59'}",   '%Y-%m-%d %H:%M')
        _all = [
            f for f in _all
            if (lambda dt: dt is None or _fs <= dt <= _fe)(_parse_file_datetime(f, FILENAME_PREFIX))
        ]

    for f in _all:
        site = _get_site_name(f, DRIVE_INPUT_DIR)
        dt   = _parse_file_datetime(f, FILENAME_PREFIX)
        all_audio_info.append({'path': f, 'site': site, 'dt': dt})

elif AUDIO_SOURCE == 'arbimon':
    from datetime import timedelta

    _AUDIO_DIR = '/content/audio'
    os.makedirs(_AUDIO_DIR, exist_ok=True)

    try:
        _jobs = JOBS
    except NameError:
        raise RuntimeError('JOBS not defined. Run Step 3 (Arbimon configuration) first.')
    if not _jobs:
        raise RuntimeError('No Arbimon sites configured. Fill in Step 3 first.')

    def _parse_arbimon_dt(filename, utc_offset_hours=0, timezone_str=None):
        from datetime import timedelta
        name = os.path.basename(filename)
        utc_dt = None
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            utc_dt = datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                              int(m.group(4)), int(m.group(5)), int(m.group(6)))
        else:
            m = re.search(r'(\d{8})_(\d{6})', name)
            if m:
                d, t = m.group(1), m.group(2)
                utc_dt = datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                                  int(t[:2]), int(t[2:4]), int(t[4:6]))
        if utc_dt is None:
            return None
        if timezone_str:
            try:
                from zoneinfo import ZoneInfo
                local_dt = utc_dt.replace(tzinfo=ZoneInfo('UTC')).astimezone(ZoneInfo(timezone_str))
                return local_dt.replace(tzinfo=None)
            except Exception:
                pass
        return utc_dt + timedelta(hours=utc_offset_hours)

    _job_lookup = {}
    for _j in _jobs:
        _job_lookup[_j['stream_id']]   = _j
        _job_lookup[_j['stream_name']] = _j

    def _resolve_job(audio_path):
        job = _job_lookup.get(os.path.basename(os.path.dirname(audio_path)))
        if job:
            return job
        for _j in _jobs:
            if _j['stream_id'] and _j['stream_id'] in audio_path:
                return _j
        return _jobs[0] if len(_jobs) == 1 else {}

    for job in _jobs:
        utc_off     = job['utc_offset_hours']
        job_utc_min = job['min_date'] - timedelta(hours=utc_off)
        job_utc_max = job['max_date'] - timedelta(hours=utc_off)
        _sep = '─' * 65
        print(f"\n{_sep}")
        print(f"Site   : {job['stream_name']}  (id: {job['stream_id']})")
        print(f"Window : {job['min_date'].strftime('%Y-%m-%d %H:%M')} → {job['max_date'].strftime('%Y-%m-%d %H:%M')}")
        _stream_dir = os.path.join(_AUDIO_DIR, job['stream_id'])
        _existing = (
            _glob.glob(f'{_stream_dir}/**/*.wav',  recursive=True) +
            _glob.glob(f'{_stream_dir}/**/*.flac', recursive=True) +
            _glob.glob(f'{_stream_dir}/**/*.mp3',  recursive=True)
        )
        if _existing:
            print(f'  {len(_existing)} file(s) already downloaded — skipping.')
        else:
            print('Downloading...')
            try:
                client.download_segments(
                    dest_path=_AUDIO_DIR,
                    stream=job['stream_id'],
                    min_date=job_utc_min,
                    max_date=job_utc_max,
                    parallel=True,
                )
            except Exception as e:
                print(f'  ERROR: {e}')

    _all = sorted(
        _glob.glob(f'{_AUDIO_DIR}/**/*.wav',  recursive=True) +
        _glob.glob(f'{_AUDIO_DIR}/**/*.flac', recursive=True) +
        _glob.glob(f'{_AUDIO_DIR}/**/*.mp3',  recursive=True)
    )

    for f in _all:
        _job = _resolve_job(f)
        site = _job.get('stream_name', os.path.basename(os.path.dirname(f)))
        dt   = _parse_arbimon_dt(
            os.path.basename(f),
            utc_offset_hours=_job.get('utc_offset_hours', 0),
            timezone_str=_job.get('timezone_str', None)
        )
        if dt is not None and _job.get('min_date') and _job.get('max_date'):
            if not (_job['min_date'] <= dt <= _job['max_date']):
                continue
        all_audio_info.append({'path': f, 'site': site, 'dt': dt})

# Summary
_sites = defaultdict(list)
for info in all_audio_info:
    _sites[info['site']].append(info)

print(f'Audio source : {AUDIO_SOURCE}')
print(f'Total files  : {len(all_audio_info)}')
print(f'Sites found  : {len(_sites)}')
print()
for site_name, entries in sorted(_sites.items()):
    dts = [e['dt'] for e in entries if e['dt'] is not None]
    if dts:
        date_range = f"{min(dts).strftime('%Y-%m-%d %H:%M')} → {max(dts).strftime('%Y-%m-%d %H:%M')}"
    else:
        date_range = '(no timestamps parsed)'
    print(f'  {site_name}')
    print(f'    Files : {len(entries)}')
    print(f'    Range : {date_range}')

if not all_audio_info:
    print('No audio files found. Check your audio source configuration in Step 2.')
else:
    print('\nScan complete. Continue to Step 5.')

---
## Step 5 — Load Model & Initialize Database

This cell downloads the ONNX backbone model (if needed) and creates the SQLite embeddings database.

If the database already exists, it validates that the model configuration matches the existing one — you cannot mix embeddings from different models in the same database.

In [ ]:
#@title 🧠 Load model & initialize database { display-mode: "form" }

import os
import sqlite3
import numpy as np
from pathlib import Path
from datetime import datetime

# --- Resolve and load ONNX model ---
if HF_REPO:
    from huggingface_hub import hf_hub_download
    print(f'Downloading backbone from HuggingFace: {HF_REPO} / {HF_FILE} ...')
    _model_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
else:
    _model_path = CUSTOM_DRIVE_MODEL_PATH

if not os.path.exists(_model_path):
    raise FileNotFoundError(f'Model not found: {_model_path}\nCheck your model configuration in Step 2.')

import onnxruntime as ort
_ort_session = ort.InferenceSession(_model_path)
_in_names    = [i.name for i in _ort_session.get_inputs()]
_out_names   = [o.name for o in _ort_session.get_outputs()]

# Validate output name; fall back to first output if not found
if OUTPUT_NAME in _out_names:
    _fetch_output = OUTPUT_NAME
else:
    _fetch_output = _out_names[0]
    print(f"WARNING: output '{OUTPUT_NAME}' not found in model; using '{_fetch_output}' instead.")

print('ONNX model loaded.')
print(f'  Path       : {_model_path}')
print(f"  Inputs     : {_in_names}  →  using '{INPUT_NAME}'")
print(f"  Outputs    : {_out_names}  →  fetching '{_fetch_output}'")
print(f'  Providers  : {_ort_session.get_providers()}')

# --- Initialize SQLite database ---
if DB_MODE == 'update existing':
    _db_path = Path(EXISTING_DB_PATH.strip())
    if not _db_path.exists():
        raise FileNotFoundError(f'Existing database not found: {_db_path}')
else:
    _db_path = Path(DRIVE_EMBEDDINGS_DIR) / f"{MODEL_NAME}.embeddings.db"
    if _db_path.exists():
        raise FileExistsError(
            f'Database already exists: {_db_path}\n'
            'Switch DB_MODE to "update existing" to add recordings to it, '
            'or delete the file to start fresh.'
        )

_meta = {
    'format_version':    '1',
    'model_id':          MODEL_NAME,
    'sample_rate':       str(SAMPLE_RATE),
    'window_duration_s': str(WINDOW_S),
    'embedding_size':    str(EMBEDDING_SIZE),
    'overlap':           str(SEGMENT_OVERLAP),
    'embedding_dtype':   'float32',
}
if HF_REPO:
    _meta['hf_repo'] = HF_REPO
    _meta['hf_file'] = HF_FILE

with sqlite3.connect(_db_path) as _con:
    _con.execute('CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL)')
    _con.execute('''
        CREATE TABLE IF NOT EXISTS embeddings (
            site_name    TEXT    NOT NULL DEFAULT '',
            recording_id TEXT    NOT NULL,
            chunk_index  INTEGER NOT NULL,
            datetime     TEXT,
            data         BLOB    NOT NULL,
            PRIMARY KEY (site_name, recording_id, chunk_index)
        )''')
    _con.execute('''
        CREATE INDEX IF NOT EXISTS idx_emb_recording
        ON embeddings(site_name, recording_id, chunk_index)''')
    _con.execute('''
        CREATE TABLE IF NOT EXISTS sites (
            site_name  TEXT PRIMARY KEY,
            stream_id  TEXT NOT NULL DEFAULT '',
            utc_offset REAL NOT NULL DEFAULT 0.0,
            lat        REAL,
            lon        REAL
        )''')
    _con.commit()

    _existing = dict(_con.execute('SELECT key, value FROM metadata').fetchall())
    _check_keys = ('model_id', 'sample_rate', 'window_duration_s', 'embedding_size', 'overlap')
    if _existing:
        _mismatch = [
            f"  {k}: DB='{_existing[k]}' vs current='{_meta[k]}'"
            for k in _check_keys
            if k in _existing and _existing[k] != _meta[k]
        ]
        if _mismatch:
            print('\nERROR: existing database was created with a different configuration:')
            for m in _mismatch:
                print(m)
            print('Use a different output folder, or delete the existing .db file to start fresh.')
        else:
            _n = _con.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]
            print(f'\nExisting database: {_n} embedding(s) already stored.')
    else:
        _con.executemany(
            'INSERT OR IGNORE INTO metadata (key, value) VALUES (?, ?)',
            list(_meta.items()) + [('created_at', datetime.now().isoformat())]
        )
        _con.commit()
        print('\nNew embeddings database created.')

print(f'\nDatabase : {_db_path}')
for k, v in _meta.items():
    print(f'  {k}: {v}')

---
## Step 6 — Check Already Computed Embeddings

This cell checks which audio files have already been processed and stored in the database.
Those files will be **skipped** in Step 7, so you can safely re-run without recomputing existing embeddings.

In [ ]:
#@title 🔎 Check already computed embeddings { display-mode: "form" }

import sqlite3
import shutil
import os
from pathlib import Path

# Work from a local copy so the check query runs fast and subsequent writes
# stay on local storage (avoids slow SQLite writes over Drive FUSE).
_local_db = Path("/content") / _db_path.name

if _db_path.exists():
    print(f"Copying database from Drive to local storage...")
    shutil.copy2(_db_path, _local_db)
    print(f"  {_db_path.stat().st_size / 1024:.1f} kB copied.")
else:
    print("WARNING: database not found on Drive — run Step 5 first.")

to_process   = []
already_done = []

with sqlite3.connect(_local_db) as _con:
    for info in all_audio_info:
        stem  = os.path.splitext(os.path.basename(info["path"]))[0]
        count = _con.execute(
            "SELECT COUNT(*) FROM embeddings WHERE site_name = ? AND recording_id = ?",
            (info['site'], stem)
        ).fetchone()[0]
        if count > 0:
            already_done.append(info)
        else:
            to_process.append(info)

print(f"Local DB         : {_local_db}")
print(f"Drive DB         : {_db_path}")
print(f"Total audio files: {len(all_audio_info)}")
print(f"Already computed : {len(already_done)}")
print(f"Remaining to run : {len(to_process)}")

if already_done:
    print()
    print("Already done (will be skipped):")
    for info in already_done[:10]:
        print(f"  {info['site']}/{os.path.basename(info['path'])}")
    if len(already_done) > 10:
        print(f"  ... and {len(already_done) - 10} more")

if not to_process:
    print()
    print("All files have already been processed. Nothing to do!")
else:
    print()
    print(f"Ready to compute embeddings for {len(to_process)} file(s). Proceed to Step 7.")

---
## Step 7 — Compute and Save Embeddings

This is the main computation step. For each audio file:
1. The recording is loaded and resampled to the model's sample rate
2. It is split into windows of the configured duration (with optional overlap)
3. Each window is fed to the ONNX backbone model
4. The resulting embedding vector is saved to the SQLite database

**Embedding keys** follow the pattern: `{site}/{filename_stem}/{start_time_s:.3f}`

> Depending on the number of files and recording length, this step may take a while. Progress is shown below.

In [ ]:
#@title 🚀 Compute embeddings { display-mode: "form" }

import os
import shutil
import sqlite3
import time
import numpy as np
import librosa
from pathlib import Path

# Sync local DB back to Drive every this many files (0 = only at the end).
SYNC_EVERY = 50  #@param {type:"integer"}

_local_db       = Path("/content") / _db_path.name
_stride_samples = max(1, int(WINDOW_SAMPLES * (1 - SEGMENT_OVERLAP)))

def _embed_segment(audio_window):
    seg = audio_window.astype(np.float32)
    if len(seg) < WINDOW_SAMPLES:
        seg = np.pad(seg, (0, WINDOW_SAMPLES - len(seg)))
    out = _ort_session.run([_fetch_output], {INPUT_NAME: seg[np.newaxis, :]})[0]
    return out.squeeze(0).astype(np.float32)

def _sync_to_drive():
    shutil.copy2(_local_db, _db_path)
    size_mb = _local_db.stat().st_size / 1024 / 1024
    print(f"  [sync] {size_mb:.1f} MB → {_db_path}")

if not to_process:
    print("No files to process. All embeddings are already computed.")
else:
    print(f"Computing embeddings for {len(to_process)} file(s)")
    print(f"  Model       : {MODEL_NAME}")
    print(f"  Sample rate : {SAMPLE_RATE} Hz")
    print(f"  Window      : {WINDOW_S} s  ({WINDOW_SAMPLES} samples)")
    print(f"  Stride      : {_stride_samples / SAMPLE_RATE:.3f} s  ({_stride_samples} samples)")
    print(f"  Embedding   : {EMBEDDING_SIZE}-dim")
    print(f"  Local DB    : {_local_db}")
    print(f"  Drive DB    : {_db_path}")
    print(f"  Drive sync  : every {SYNC_EVERY} file(s)")
    print("=" * 65)

    total_segments = 0
    total_start    = time.time()


    # Build per-site metadata (stream_id, UTC offset, coordinates)
    _site_meta = {}
    if AUDIO_SOURCE == 'arbimon':
        _jobs_by_key = {}
        try:
            for _j in JOBS:
                _jobs_by_key[_j['stream_name']] = _j
                _jobs_by_key[_j['stream_id']]   = _j
        except NameError:
            pass
        for _info in to_process:
            _sn = _info['site']
            if _sn not in _site_meta:
                _j = _jobs_by_key.get(_sn, {})
                _site_meta[_sn] = {
                    'stream_id':  _j.get('stream_id', ''),
                    'utc_offset': float(_j.get('utc_offset_hours', 0)),
                    'lat':        _j.get('lat', '') or '',
                    'lon':        _j.get('lon', '') or '',
                }
    else:
        for _info in to_process:
            _sn = _info['site']
            if _sn not in _site_meta:
                _coords = _latlon_map.get(_sn, ('', ''))
                _site_meta[_sn] = {
                    'stream_id':  '',
                    'utc_offset': 0.0,
                    'lat':        _coords[0] if _coords[0] != '' else None,
                    'lon':        _coords[1] if _coords[1] != '' else None,
                }
    with sqlite3.connect(_local_db) as _con:
        _con.execute("PRAGMA journal_mode=MEMORY")
        _con.execute("PRAGMA synchronous=NORMAL")
        _con.execute('''
            CREATE TABLE IF NOT EXISTS sites (
                site_name  TEXT PRIMARY KEY,
                stream_id  TEXT NOT NULL DEFAULT '',
                utc_offset REAL NOT NULL DEFAULT 0.0,
                lat        REAL,
                lon        REAL
            )''')

        for _sn, _sm in _site_meta.items():
            _lat = float(_sm['lat']) if _sm['lat'] not in ('', None) else None
            _lon = float(_sm['lon']) if _sm['lon'] not in ('', None) else None
            _con.execute(
                'INSERT OR IGNORE INTO sites (site_name, stream_id, utc_offset, lat, lon) VALUES (?, ?, ?, ?, ?)',
                (_sn, _sm['stream_id'], _sm['utc_offset'], _lat, _lon)
            )

        for file_idx, info in enumerate(to_process, 1):
            audio_path = info["path"]
            site_name  = info["site"]
            filename   = os.path.basename(audio_path)
            file_stem  = os.path.splitext(filename)[0]

            print(f"[{file_idx}/{len(to_process)}] {filename}  (site: {site_name})")

            try:
                audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
            except Exception as e:
                print(f"  ERROR loading audio: {e} — skipping.")
                continue

            file_start   = time.time()
            n_segments   = 0
            rec_datetime = info['dt'].isoformat() if info['dt'] else None
            chunk_index  = 0

            for start_samp in range(0, len(audio), _stride_samples):
                seg = audio[start_samp:start_samp + WINDOW_SAMPLES]
                if len(seg) < WINDOW_SAMPLES * 0.5:
                    continue
                start_s = start_samp / SAMPLE_RATE
                try:
                    emb = _embed_segment(seg)
                except Exception as e:
                    print(f"  ERROR during inference at {start_s:.1f}s: {e}")
                    continue
                _con.execute(
                    "INSERT OR REPLACE INTO embeddings "
                    "(site_name, recording_id, chunk_index, datetime, data) VALUES (?, ?, ?, ?, ?)",
                    (site_name, file_stem, chunk_index, rec_datetime, emb.tobytes())
                )
                chunk_index += 1
                n_segments  += 1

            _con.commit()
            total_segments += n_segments
            elapsed     = time.time() - file_start
            audio_dur_s = len(audio) / SAMPLE_RATE
            per_seg_ms  = elapsed / n_segments * 1000 if n_segments else 0
            print(f"  → {n_segments} segments  |  {elapsed:.1f}s  "
                  f"({per_seg_ms:.0f} ms/seg, {audio_dur_s / elapsed:.1f}x realtime)")

            if SYNC_EVERY > 0 and file_idx % SYNC_EVERY == 0:
                _sync_to_drive()

    # Final sync to Drive
    _sync_to_drive()

    total_elapsed = time.time() - total_start
    print()
    print("=" * 65)
    print("Embedding extraction complete!")
    print(f"  Files processed  : {len(to_process)}")
    print(f"  Total segments   : {total_segments}")
    print(f"  Total time       : {total_elapsed:.1f}s")
    print(f"  Database         : {_db_path}")
    print(f"  DB size          : {_db_path.stat().st_size / 1024 / 1024:.2f} MB")